# Races and predictions

Owner-run, local-only view of Garmin race flags, local overrides and cached predictions. Outputs may contain private activity data; never commit or share them.

In [ ]:
import polars as pl

from stridelog.db.models import Activity, ActivityAnalytics, ActivityAnnotation
from stridelog.notebook_data import read_table

In [ ]:
activities = {
    row["id"]: row
    for row in read_table(Activity, ["id", "name", "local_date", "is_race", "distance_m"])
}
annotations = {
    row["activity_id"]: row
    for row in read_table(
        ActivityAnnotation, ["activity_id", "is_race_override", "race_name", "official_time_s"]
    )
}
cache = read_table(ActivityAnalytics, ["activity_id", "metrics"])
rows = []
for item in cache:
    activity = activities.get(item["activity_id"])
    if activity is None:
        continue
    annotation = annotations.get(item["activity_id"], {})
    is_race = (
        activity["is_race"]
        if annotation.get("is_race_override") is None
        else annotation["is_race_override"]
    )
    if is_race:
        rows.append(
            {
                "name": annotation.get("race_name") or activity["name"],
                "local_date": activity["local_date"],
                "distance_m": activity["distance_m"],
                "official_time_s": annotation.get("official_time_s"),
                "predictions": item["metrics"].get("race_predictions", []),
            }
        )
pl.DataFrame(rows)